# Goal: statistical-summary 

In [1]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
Date: 10-10-2026 14:39 IST
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\nDate: 10-10-2026 14:39 IST\n'

In [2]:
import polars as pl

In [3]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-07\data\lev-07_merged.parquet"
pdf = pl.scan_parquet(path)

In [4]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('Kerosene_ration_card', String),
        ('LPG_subsidy_received', String),
        ('LPG_subsidized_cylinders', String),
        ('Free_electricity', String),
        ('Any_member_attended_school', String),
        ('Num_govt_school_attended', Float64),
        ('Num_private_school_attended', Float64),
        ('Free_textbooks_received', String),
        ('Total_textbooks', Float64),
       

# Useful Variables

In [5]:
lev_07_food = [
    'Free_other_items_received',
    'Total_other_items',
    'Fee_waiver_received',
    'Num_fee_waiver_received',
]

lev_07_health = [
    'Ayushman_beneficiary',
    'Num_ayushman_beneficiaries',
    'Hospitalization_case',
    'Medical_benefit_received',
    'Num_medical_beneficiaries',
    'Medical_benefit_amount',
]

lev_07_digital = [
    'Online_purchase_fuel_light',
    'Online_purchase_toilet_articles',
    'Online_purchase_education',
    'Online_purchase_medicine',
    'Online_purchase_services',
    'Multiplier'
]

lev_07_occupation = [
    'Any_member_attended_school',
    'Num_govt_school_attended',
    'Num_private_school_attended',
    'Free_textbooks_received',
    'Total_textbooks',
    'Free_stationery_received',
    'Total_stationery',
    'Free_school_bag_received',
    'Total_school_bags',
]

lev_07_govt = [
    'Kerosene_ration_card',
    'LPG_subsidy_received',
    'LPG_subsidized_cylinders',
    'Free_electricity',
]

In [6]:
lev_07_cols = (
    lev_07_food
    + lev_07_health
    + lev_07_digital
    + lev_07_occupation
    + lev_07_govt
)

In [7]:
df = pdf.select(lev_07_cols)

In [8]:
df.head(2).collect()

Free_other_items_received,Total_other_items,Fee_waiver_received,Num_fee_waiver_received,Ayushman_beneficiary,Num_ayushman_beneficiaries,Hospitalization_case,Medical_benefit_received,Num_medical_beneficiaries,Medical_benefit_amount,Online_purchase_fuel_light,Online_purchase_toilet_articles,Online_purchase_education,Online_purchase_medicine,Online_purchase_services,Multiplier,Any_member_attended_school,Num_govt_school_attended,Num_private_school_attended,Free_textbooks_received,Total_textbooks,Free_stationery_received,Total_stationery,Free_school_bag_received,Total_school_bags,Kerosene_ration_card,LPG_subsidy_received,LPG_subsidized_cylinders,Free_electricity
str,f64,str,f64,str,f64,str,str,f64,f64,f64,f64,f64,f64,f64,i64,str,f64,f64,str,f64,str,f64,str,f64,str,str,str,str
"""""",null,"""2""",null,"""1""",1.0,"""4""","""""",null,null,null,null,null,null,null,57986,"""1""",0.0,3.0,"""""",null,"""""",null,"""""",null,"""2""","""1""","""1""","""2"""
"""""",null,"""2""",null,"""2""",null,"""2""","""2""",null,null,null,null,null,null,1.0,57986,"""1""",0.0,2.0,"""""",null,"""""",null,"""""",null,"""2""","""1""","""2""","""2"""


In [9]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in lev_07_cols]
)

In [10]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

Free_other_items_received,Total_other_items,Fee_waiver_received,Num_fee_waiver_received,Ayushman_beneficiary,Num_ayushman_beneficiaries,Hospitalization_case,Medical_benefit_received,Num_medical_beneficiaries,Medical_benefit_amount,Online_purchase_fuel_light,Online_purchase_toilet_articles,Online_purchase_education,Online_purchase_medicine,Online_purchase_services,Multiplier,Any_member_attended_school,Num_govt_school_attended,Num_private_school_attended,Free_textbooks_received,Total_textbooks,Free_stationery_received,Total_stationery,Free_school_bag_received,Total_school_bags,Kerosene_ration_card,LPG_subsidy_received,LPG_subsidized_cylinders,Free_electricity
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
2,33,3,10,2,19,4,3,10,549,2,2,2,2,2,23567,2,11,13,2,59,2,43,2,10,2,2,4,2


# Logic

In [11]:
categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_21112\2508435801.py:4: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


# Numerical Columns

In [12]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
Total_other_items,26858.0,497048.0,2.884504,2.951695,1.0,1.0,2.0,4.0,50.0
Num_ayushman_beneficiaries,172338.0,351568.0,3.314405,1.652805,1.0,2.0,3.0,4.0,20.0
Medical_benefit_amount,9336.0,514570.0,28576.167952,54936.122846,0.0,4500.0,12500.0,30000.0,1000000.0
Multiplier,523906.0,0.0,111344.812944,79656.036356,369.0,56738.0,114044.0,150441.0,2366902.0
Num_private_school_attended,206340.0,317566.0,1.127673,0.999558,0.0,0.0,1.0,2.0,12.0
Total_textbooks,108656.0,415250.0,9.390296,5.388385,1.0,6.0,8.0,12.0,84.0
Total_stationery,23040.0,500866.0,6.120399,5.356625,1.0,3.0,4.0,8.0,50.0


# Categorical Columns

In [13]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

Free_other_items_received


Free_other_items_received,count
i32,u32
null,497048
1,26858


Fee_waiver_received


Fee_waiver_received,count
i32,u32
2,251912
null,238838
1,33156


Num_fee_waiver_received


...

LPG_subsidy_received,count
i32,u32
2,365286
1,158620


LPG_subsidized_cylinders


LPG_subsidized_cylinders,count
i32,u32
null,365286
1,89156
2,48580
3,20884


Free_electricity


Free_electricity,count
i32,u32
2,411154
1,112752
